# TP1 
**Encadré par : Mustapha Ghiati**

Ce TP combine les étapes : preprocessing, encodage, pipeline et modélisation.

## 1. Import des librairies

In [14]:
import pandas as pd
import numpy as np
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

## 2. Chargement des données

In [15]:
df = sns.load_dataset("diamonds")
df.head()

,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


## 3. Exploration des données

In [16]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53940 entries, 0 to 53939
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype   
---  ------   --------------  -----   
 0   carat    53940 non-null  float64 
 1   cut      53940 non-null  category
 2   color    53940 non-null  category
 3   clarity  53940 non-null  category
 4   depth    53940 non-null  float64 
 5   table    53940 non-null  float64 
 6   price    53940 non-null  int64   
 7   x        53940 non-null  float64 
 8   y        53940 non-null  float64 
 9   z        53940 non-null  float64 
dtypes: category(3), float64(6), int64(1)
memory usage: 3.0 MB


,carat,depth,table,price,x,y,z
count,53940.000000,53940.000000,53940.000000,53940.000000,53940.000000,53940.000000,53940.000000
mean,0.797940,61.749405,57.457184,3932.799722,5.731157,5.734526,3.538734
std,0.474011,1.432621,2.234491,3989.439738,1.121761,1.142135,0.705699
min,0.200000,43.000000,43.000000,326.000000,0.000000,0.000000,0.000000
25%,0.400000,61.000000,56.000000,950.000000,4.710000,4.720000,2.910000
50%,0.700000,61.800000,57.000000,2401.000000,5.700000,5.710000,3.530000
75%,1.040000,62.500000,59.000000,5324.250000,6.540000,6.540000,4.040000
max,5.010000,79.000000,95.000000,18823.000000,10.740000,58.900000,31.800000


## 4. Train / Test split

In [17]:
train_set, test_set = train_test_split(df,test_size=0.2, random_state=0)

In [18]:
print("train_set :",train_set.shape)
print("test_set :",test_set.shape)

train_set : (43152, 10)
test_set : (10788, 10)


## 5. Encodage


Les variables catégorielles ont été analysées.
Les variables cut et clarity présentent un ordre naturel, donc un encodage ordinal est utilisé.
La variable color ne présente pas d’ordre significatif, donc un encodage One-Hot est appliqué.

In [19]:
# Déterminer les différentes valeurs qui se cachent dans la première colonne catégorielle
valeurs_cut = train_set["cut"].unique()
print(valeurs_cut)

['Ideal', 'Premium', 'Very Good', 'Good', 'Fair']
Categories (5, object): ['Ideal', 'Premium', 'Very Good', 'Good', 'Fair']


In [20]:
# Déterminer les différentes valeurs qui se cachent dans la deuxième colonne catégorielle
valeurs_color = train_set["color"].unique()
print(valeurs_color)

['G', 'E', 'H', 'D', 'F', 'I', 'J']
Categories (7, object): ['D', 'E', 'F', 'G', 'H', 'I', 'J']


In [21]:
# Déterminer les différentes valeurs qui se cachent dans  la troisième colonne catégorielle
valeurs_clarity = train_set["clarity"].unique()
print(valeurs_clarity)

['VS1', 'VS2', 'VVS2', 'SI1', 'IF', 'VVS1', 'SI2', 'I1']
Categories (8, object): ['IF', 'VVS1', 'VVS2', 'VS1', 'VS2', 'SI1', 'SI2', 'I1']


In [22]:
cut_order = ['Fair', "Good", "Very Good", "Premium", "Ideal"]
color_order = ["D", "E", "F", "G", "H", "I", "J"]
clarity_order = ['I1', 'SI2', 'SI1', 'VS2', 'VS1', 'VVS2', 'VVS1', 'IF']
#Introduire cut_order color_order dans l'Encoder
encoder = OrdinalEncoder(categories=[cut_order, color_order, clarity_order])
encoding_results = encoder.fit_transform(train_set[['cut', 'color', 'clarity']])
encoding_results

array([[4., 3., 4.],
       [4., 3., 3.],
       [4., 1., 5.],
       ...,
       [3., 5., 4.],
       [4., 3., 7.],
       [4., 2., 1.]], shape=(43152, 3))

In [23]:
train_set

,carat,cut,color,clarity,depth,table,price,x,y,z
26250,1.63,Ideal,G,VS1,61.7,55.0,15697,7.56,7.60,4.68
31510,0.34,Ideal,G,VS2,62.2,57.0,765,4.47,4.44,2.77
40698,0.40,Ideal,E,VVS2,61.7,56.0,1158,4.73,4.77,2.93
42634,0.58,Premium,H,SI1,62.1,55.0,1332,5.38,5.35,3.33
47714,0.63,Very Good,D,SI1,62.8,57.0,1885,5.40,5.46,3.41
...,...,...,...,...,...,...,...,...,...,...
45891,0.52,Premium,F,VS2,60.7,59.0,1720,5.18,5.14,3.13
52416,0.70,Good,D,SI1,63.6,60.0,2512,5.59,5.51,3.51
42613,0.32,Premium,I,VS1,61.3,58.0,505,4.35,4.39,2.68
43567,0.41,Ideal,G,IF,61.0,57.0,1431,4.81,4.79,2.93


In [24]:
# Injecter le résultat de l'encodage dans le tableau de trainset

train_set[['cut', 'color', 'clarity']] = encoding_results

train_set

,carat,cut,color,clarity,depth,table,price,x,y,z
26250,1.63,4.0,3.0,4.0,61.7,55.0,15697,7.56,7.60,4.68
31510,0.34,4.0,3.0,3.0,62.2,57.0,765,4.47,4.44,2.77
40698,0.40,4.0,1.0,5.0,61.7,56.0,1158,4.73,4.77,2.93
42634,0.58,3.0,4.0,2.0,62.1,55.0,1332,5.38,5.35,3.33
47714,0.63,2.0,0.0,2.0,62.8,57.0,1885,5.40,5.46,3.41
...,...,...,...,...,...,...,...,...,...,...
45891,0.52,3.0,2.0,3.0,60.7,59.0,1720,5.18,5.14,3.13
52416,0.70,1.0,0.0,2.0,63.6,60.0,2512,5.59,5.51,3.51
42613,0.32,3.0,5.0,4.0,61.3,58.0,505,4.35,4.39,2.68
43567,0.41,4.0,3.0,7.0,61.0,57.0,1431,4.81,4.79,2.93


## 6. Normalisation

In [25]:
from sklearn.preprocessing import MinMaxScaler

In [26]:
scaler=MinMaxScaler()
scaled_data=scaler.fit_transform(train_set)
train_set # train_set has the same values as scaled_data

,carat,cut,color,clarity,depth,table,price,x,y,z
26250,1.63,4.0,3.0,4.0,61.7,55.0,15697,7.56,7.60,4.68
31510,0.34,4.0,3.0,3.0,62.2,57.0,765,4.47,4.44,2.77
40698,0.40,4.0,1.0,5.0,61.7,56.0,1158,4.73,4.77,2.93
42634,0.58,3.0,4.0,2.0,62.1,55.0,1332,5.38,5.35,3.33
47714,0.63,2.0,0.0,2.0,62.8,57.0,1885,5.40,5.46,3.41
...,...,...,...,...,...,...,...,...,...,...
45891,0.52,3.0,2.0,3.0,60.7,59.0,1720,5.18,5.14,3.13
52416,0.70,1.0,0.0,2.0,63.6,60.0,2512,5.59,5.51,3.51
42613,0.32,3.0,5.0,4.0,61.3,58.0,505,4.35,4.39,2.68
43567,0.41,4.0,3.0,7.0,61.0,57.0,1431,4.81,4.79,2.93


## 7. Transformation du test set

In [27]:
test_set[['cut', 'color', 'clarity']]

,cut,color,clarity
10176,Ideal,H,SI2
16083,Ideal,H,SI1
13420,Premium,I,SI1
20407,Ideal,F,SI1
8909,Very Good,F,VS2
...,...,...,...
42208,Good,H,VS2
3638,Very Good,G,SI2
5508,Very Good,F,SI2
19535,Ideal,G,VVS1


In [28]:
encoding_results = encoder.transform(test_set[['cut', 'color', 'clarity']])
encoding_results

array([[4., 4., 1.],
       [4., 4., 2.],
       [3., 5., 2.],
       ...,
       [2., 2., 1.],
       [4., 3., 6.],
       [1., 1., 5.]], shape=(10788, 3))

In [29]:
test_set[['cut', 'color', 'clarity']]=encoding_results
test_set

,carat,cut,color,clarity,depth,table,price,x,y,z
10176,1.10,4.0,4.0,1.0,62.0,55.0,4733,6.61,6.65,4.11
16083,1.29,4.0,4.0,2.0,62.6,56.0,6424,6.96,6.93,4.35
13420,1.20,3.0,5.0,2.0,61.1,58.0,5510,6.88,6.80,4.18
20407,1.50,4.0,2.0,2.0,60.9,56.0,8770,7.43,7.36,4.50
8909,0.90,2.0,2.0,3.0,61.7,57.0,4493,6.17,6.21,3.82
...,...,...,...,...,...,...,...,...,...,...
42208,0.52,1.0,4.0,3.0,63.6,57.0,1289,5.05,5.10,3.23
3638,0.91,2.0,3.0,1.0,60.4,61.0,3435,6.21,6.28,3.77
5508,1.08,2.0,2.0,1.0,63.4,55.0,3847,6.53,6.50,4.13
19535,1.02,4.0,3.0,6.0,61.5,57.0,8168,6.44,6.47,3.97


In [30]:
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer 

In [31]:
df = sns.load_dataset("diamonds")
df.head()

,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


In [32]:
train_set, test_set = train_test_split(df,test_size=0.2,random_state=0)

In [33]:
cat_cols=["cut", "color", "clarity"]
num_cols=[c for c in df.columns if c not in cat_cols]

In [34]:
#avant dappliquer lencodage ordinal, il faut spécifiez l'ordre de chaque catégorie
cut_order = ["Fair", "Good", "Very Good", "Premium", "Ideal"]
color_order = ["J", "I", "H", "G", "F", "E", "D"]
clarity_order = ["I1", "SI2", "SI1", "VS2", "VS1", "VVS2", "VVS1", "IF"]
ordinal = OrdinalEncoder(categories=[cut_order, color_order, clarity_order])

preprocess=ColumnTransformer(
    transformers=[
        ("ord", ordinal, cat_cols),
        ("num", StandardScaler(), num_cols),
    ],
    remainder="passthrough",
)



In [35]:
pipeline = Pipeline([("prep",preprocess)])

In [36]:
pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('prep', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('ord', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse mat

In [37]:
X_train_transformed=pipeline.fit_transform(train_set)

In [38]:
X_train_transformed

array([[ 4.        ,  3.        ,  4.        , ...,  1.62844575,
         1.62183518,  1.60909161],
       [ 4.        ,  3.        ,  3.        , ..., -1.12336525,
        -1.12610585, -1.08459534],
       [ 4.        ,  5.        ,  5.        , ..., -0.89182128,
        -0.83913732, -0.85894617],
       ...,
       [ 3.        ,  1.        ,  4.        , ..., -1.2302317 ,
        -1.16958593, -1.211523  ],
       [ 4.        ,  3.        ,  7.        , ..., -0.82057699,
        -0.82174529, -0.85894617],
       [ 4.        ,  4.        ,  1.        , ...,  0.45291484,
         0.39569694,  0.36802118]], shape=(43152, 10))

In [39]:
pipeline.transform(test_set)

array([[ 4.        ,  2.        ,  1.        , ...,  0.78241971,
         0.79571367,  0.80521645],
       [ 4.        ,  2.        ,  2.        , ...,  1.09411352,
         1.03920211,  1.1436902 ],
       [ 3.        ,  1.        ,  2.        , ...,  1.02286922,
         0.92615391,  0.90393796],
       ...,
       [ 2.        ,  4.        ,  1.        , ...,  0.71117542,
         0.66527343,  0.83342259],
       [ 4.        ,  3.        ,  6.        , ...,  0.63102558,
         0.63918538,  0.60777342],
       [ 1.        ,  5.        ,  5.        , ..., -0.67808839,
        -0.63912896, -0.43585398]], shape=(10788, 10))

In [50]:
from sklearn.model_selection import train_test_split

X = df.drop("price", axis=1)
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [51]:
model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('ord', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different trans

## 8. Évaluation

In [55]:
from sklearn.metrics import mean_squared_error
import numpy as np

# Prédiction
y_pred = model.predict(X_test)

# Évaluation
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("RMSE :", rmse)

RMSE : 550.4171916020578


## Exercice : Création des pipelines

In [70]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.metrics import mean_squared_error, r2_score

In [59]:
df = sns.load_dataset("diamonds")
df.head()

,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


In [60]:
X = df.drop("price", axis=1)
y = df["price"]

In [61]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [62]:
categorical_cols = ["cut", "color", "clarity"]

In [63]:
cut_order = ["Fair", "Good", "Very Good", "Premium", "Ideal"]
color_order = ["J", "I", "H", "G", "F", "E", "D"]
clarity_order = ["I1", "SI2", "SI1", "VS2", "VS1", "VVS2", "VVS1", "IF"]

In [64]:
column_transformer = ColumnTransformer(
    transformers=[
        ("Encoder", OrdinalEncoder(categories=[cut_order, color_order, clarity_order]), categorical_cols)
    ],
    remainder="passthrough"
)

In [65]:
pipeline = Pipeline(steps=[
    ("Encoder", column_transformer),
    ("Scaler", MinMaxScaler())
])

In [66]:
advanced_pipeline = Pipeline(steps=[
    ("preprocessing", pipeline),
    ("FeatureEngineering", PolynomialFeatures(degree=2, include_bias=False)),
    ("SelectFeatures", SelectKBest(score_func=f_regression, k=10)),
    ("model", RandomForestRegressor(random_state=42))
])

In [67]:
advanced_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('FeatureEngineering', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('Encoder', ...), ('Scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'d

In [68]:
y_pred = advanced_pipeline.predict(X_test)

In [71]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("RMSE :", rmse)
print("R2 :", r2)

RMSE : 1372.0915917686611
R2 : 0.8815715870009294


Un pipeline avancé a été mis en place intégrant une étape de génération de nouvelles caractéristiques avec PolynomialFeatures, suivie d’une sélection des variables les plus pertinentes avec SelectKBest.
Ce pipeline permet d’améliorer la performance du modèle en réduisant le bruit et en capturant des relations non linéaires.

## Conclusion

- Utilisation combinée de plusieurs types d'encodage.
- Pipeline complet pour automatiser preprocessing + modèle.
- Bonne structuration inspirée des deux TP.